# 84장 · SQLite 스키마 변경과 부분 업데이트

우선순위를 추가하기 전에 만든 행을 새 화면에서 열면 마감일과 우선순위에 무엇이 보일까요?

기존 투두 데이터에 마감일과 우선순위를 추가합니다. SQL 이관 직후의 옛 행과 각 필드를 수정한 뒤의 값을 비교해 누락·초기화 오류를 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## ALTER TABLE과 기존 행의 기본값

새로 만들 때와 예전 행을 읽을 때 모두 due_date는 비어 있을 수 있고 priority의 기본값은 normal입니다. PostgreSQL에서는 날짜 타입을 사용합니다. 여기서는 SQLite의 ISO 날짜 문자열과 CHECK 조건으로 같은 동작을 좁혀 봅니다.


### 기존 세 열에 추가하는 두 필드

<p>due_date는 날짜가 정해지지 않았으면 NULL로 둘 수 있습니다. priority에는 low, normal, high만 들어갈 수 있으며 기본값은 normal입니다. 추가 직후 1번 행의 제목과 완료는 그대로 남습니다.</p>


### 셀 01 · ALTER TABLE로 열 추가

변경 전의 복습 한 건을 넣고 ADD COLUMN을 적용합니다.


In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT NOT NULL, completed INTEGER NOT NULL)")
db.execute("INSERT INTO todos VALUES(1, '복습', 0)")
db.execute("ALTER TABLE todos ADD COLUMN due_date TEXT")
db.execute("ALTER TABLE todos ADD COLUMN priority TEXT NOT NULL DEFAULT 'normal' CHECK(priority IN ('low','normal','high'))")
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습', None, 'normal')


변경 전부터 있던 1번 행에서 새 날짜는 None, 우선순위는 normal로 읽힙니다. 기존 행을 화면에 표시할 때에는 마감일을 비워 둔 상태도 처리해야 합니다.


### ALTER TABLE 이후 기존 행 읽기

<p>먼저 기존 형식으로 복습 한 행을 넣은 다음 열을 추가합니다. 기존 행을 다시 조회하면 새 due_date는 None, 기본값이 있는 priority는 normal입니다.</p>


## 값 바인딩과 수정 필드 검증

마감일과 우선순위가 함께 온 경우에는 두 열만 고치는 고정 SQL을 선택할 수 있습니다. 입력에서 직접 열 이름을 만들어 붙이지 않습니다. 한 필드가 누락된 요청과 일부러 null을 보내 날짜를 지우는 요청도 구분합니다.


### 날짜 문자열을 바인딩하기 전에 확인하기

<p>date.fromisoformat으로 실제 달력 날짜인지 검사한 뒤 isoformat 결과를 TEXT 열에 저장합니다. 이 셀에서 날짜를 검사하는 곳은 Python입니다.</p>


### 셀 02 · UPDATE의 값 바인딩

날짜 문자열은 실제 달력 날짜인지 확인한 뒤 SQL 자리표시자에 넣습니다.

날짜와 우선순위를 저장한 다음, 같은 행의 제목만 바꿔 보세요. 제목 변경 뒤에도 날짜는 2027-04-02, 우선순위는 high여야 합니다. 다음 과제에서 날짜를 직접 비울 때와 비교합니다.


In [2]:
from datetime import date
due = date.fromisoformat("2027-04-02").isoformat()
with db:
    db.execute("UPDATE todos SET due_date=?, priority=? WHERE id=?", (due, "high", 1))
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습', '2027-04-02', 'high')


복습·2027-04-02·high가 나옵니다. 바인딩 값은 SQL에 적힌 열 순서와 같습니다. 이 SQL은 마감일과 우선순위 두 열을 명시합니다. 다음 셀에서는 제목만 지정해 나머지 값이 보존되는지 확인합니다.


#### 일부 필드만 수정하는 요청

<mark>보내지 않은 필드와 명시적으로 비울 필드를 구별해야 합니다.</mark> 기한만 수정했을 때 기존 우선순위가 사라지는지 확인하세요. 열을 추가하기 전에 만들어 둔 행에서도 같은 요청을 실행하면 이전 데이터의 기본값까지 점검할 수 있습니다.

선수 개념 · 82장 SQLite 영속성과 instr() 부분 문자열 검색

### 부분 수정 시 기존 값 보존

아래 셀을 실행한 뒤 바로 앞의 결과와 비교하세요.


In [3]:
with db:
    db.execute("UPDATE todos SET title=? WHERE id=?", ("복습 완료 예정", 1))
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습 완료 예정', '2027-04-02', 'high')


UPDATE에 title만 적었습니다. 날짜와 우선순위를 다시 읽어 두 값이 유지되는지 대조하세요.


### UPDATE에서 생략한 열과 NULL을 넣은 열

<p>제목만 SET 하면 기존 기한과 우선순위가 유지됩니다. 기한을 지우려면 due_date를 SET 대상에 포함하고 None을 전달합니다. 필드를 건너뛴 동작과 값을 지운 동작을 출력에서 구별하세요.</p>


### 과제 03 · NULL과 필드 생략 구분

1번 할 일의 마감일만 지우세요. 변경하기 직전의 날짜와 변경 뒤의 날짜·우선순위를 조회해, 열을 생략한 이전 UPDATE와 비교하세요.


In [4]:
# 날짜를 먼저 조회하고, due_date만 NULL로 갱신한 뒤 같은 행을 다시 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>query = &quot;SELECT due_date, priority FROM todos WHERE id=?&quot;
before = db.execute(query, (1,)).fetchone()
with db:
    db.execute(&quot;UPDATE todos SET due_date=? WHERE id=?&quot;, (None, 1))
after = db.execute(query, (1,)).fetchone()
print(before)
print(after)</code></pre>
<p>실행 결과</p><pre><code>(&#x27;2027-04-02&#x27;, &#x27;high&#x27;)
(None, &#x27;high&#x27;)</code></pre>
<p>앞 셀의 제목 변경에서는 SET에 due_date가 없어 날짜가 남았습니다. 여기서는 None을 명시해 그 값만 NULL로 바꿨습니다. 다시 비교할 때는 날짜를 입력한 셀부터 실행하세요.</p>
</details>


#### 허용하지 않는 우선순위가 저장되지 않는 경로

```mermaid
flowchart TD
nbefore["기존 행"]
  nwrite["UPDATE"]
  ncheck["CHECK"]
  nerror["예외 처리"]
  nread["SELECT"]
  nbefore --> nwrite
  nwrite --> ncheck
  ncheck --> nerror
  nerror --> nread
```

앞에서 high로 정한 1번 행을 urgent로 바꾸려고 시도합니다. UPDATE의 입력값과 실패 후 SELECT의 값을 비교하세요.

### 셀 04 · 허용 우선순위 검증

셀 04는 priority에 urgent를 쓰고 같은 행을 다시 읽습니다. 예외 메시지와 조회한 값을 함께 보면 실패한 변경이 저장됐는지 판별할 수 있습니다.


In [5]:
try:
    with db:
        db.execute("UPDATE todos SET priority=? WHERE id=?", ("urgent", 1))
except sqlite3.IntegrityError:
    print("허용하지 않는 우선순위")
print(db.execute("SELECT priority FROM todos WHERE id=1").fetchone()[0])

허용하지 않는 우선순위
high


오류 뒤에도 high가 남습니다. 코드 검토에서는 기본값뿐 아니라 잘못된 입력을 모두 검사한 뒤 한 번에 반영하는지 살펴보세요.


### CHECK 실패 후 저장값 확인하기

<p>urgent는 허용 목록에 없어 IntegrityError가 발생합니다. except 뒤에서 같은 행의 priority를 다시 읽으면 실패한 문자열 대신 직전의 high가 남습니다.</p>


## PATCH에서 생략한 날짜와 지운 날짜 구분하기

기존 수업에서는 SQL로 마감일을 설정하고 다른 열을 갱신했습니다. 이번에는 화면이 보낸 JSON에서 날짜 필드가 빠진 경우와 명시적으로 null이 들어온 경우를 구분해 저장합니다.

### 셀 07 · 두 요청에서 실제로 보낸 필드 읽기

두 JSON 모두 우선순위는 high입니다. 하나에는 날짜가 없고 다른 하나에는 null이 있습니다. Pydantic의 exclude_unset으로 어떤 필드를 보냈는지 비교하세요.

In [6]:
from datetime import date
from typing import Literal
from pydantic import BaseModel, ConfigDict

class TodoPatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    due_date: date | None = None
    priority: Literal["low", "normal", "high"] = "normal"

kept = TodoPatch(priority="high")
cleared = TodoPatch(priority="high", due_date=None)
print(kept.model_dump(mode="json", exclude_unset=True))
print(cleared.model_dump(mode="json", exclude_unset=True))

{'priority': 'high'}
{'due_date': None, 'priority': 'high'}


첫 줄에는 priority만, 둘째 줄에는 due_date와 값 None도 들어 있습니다. 모델 속성만 읽으면 두 요청 모두 날짜가 None으로 보이므로, 어느 요청이 지우기를 선택했는지 놓칠 수 있습니다.

### 과제 08 · 보낸 열만 갱신해 두 요청을 같은 행에서 비교하기

이 장 셀 01에서 만든 db에 두 열이 추가된 상태로 시작합니다. 실습용 DB 복사본에 기준 날짜 2027-04-02를 설정합니다. 모델이 받은 필드만 UPDATE하도록 함수를 작성해 kept, cleared 순서로 적용하고 매번 저장된 날짜·우선순위를 읽으세요.

In [7]:
import sqlite3
patch_db = sqlite3.connect(":memory:")
db.backup(patch_db)
with patch_db:
    patch_db.execute("UPDATE todos SET due_date=?, priority=? WHERE id=1", ("2027-04-02", "normal"))
# 고정한 두 UPDATE와 모델이 받은 필드 목록을 사용해 갱신 함수를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import sqlite3
patch_db = sqlite3.connect(&quot;:memory:&quot;)
db.backup(patch_db)
with patch_db:
    patch_db.execute(&quot;UPDATE todos SET due_date=?, priority=? WHERE id=1&quot;, (&quot;2027-04-02&quot;, &quot;normal&quot;))

def apply_patch_values(connection, todo_id, patch):
    fields = patch.model_dump(mode=&quot;json&quot;, exclude_unset=True)
    with connection:
        if &quot;due_date&quot; in fields:
            connection.execute(&quot;UPDATE todos SET due_date=? WHERE id=?&quot;, (fields[&quot;due_date&quot;], todo_id))
        if &quot;priority&quot; in fields:
            connection.execute(&quot;UPDATE todos SET priority=? WHERE id=?&quot;, (fields[&quot;priority&quot;], todo_id))
    return connection.execute(&quot;SELECT due_date, priority FROM todos WHERE id=?&quot;, (todo_id,)).fetchone()

print(apply_patch_values(patch_db, 1, kept))
print(apply_patch_values(patch_db, 1, cleared))</code></pre>
<p>실행 결과</p><pre><code>(&#x27;2027-04-02&#x27;, &#x27;high&#x27;)
(None, &#x27;high&#x27;)</code></pre>
<p>첫 요청 뒤에는 날짜가 유지되고 high로 바뀝니다. 두 번째에는 날짜가 None이 됩니다. 두 열은 한 트랜잭션 안에서 바뀌며 사용자 입력으로 SQL 열 이름을 조립하지 않습니다.</p>
</details>

<mark>날짜 키가 빠진 요청은 유지, 날짜 키의 값이 null인 요청은 삭제입니다.</mark> 두 요청을 똑같이 보내도록 화면 코드를 짜면 사용자가 우선순위만 바꿔도 마감일이 사라질 수 있습니다. 저장 함수에 도달하기 전 브라우저의 요청 본문부터 확인하세요.

85장의 배포 확인에는 변경 전 항목 하나를 남겨 두고, 새 앱이 우선순위를 수정한 뒤에도 해당 항목의 날짜를 읽는지 포함하세요.

## 돌아보기

마감일을 지우고도 high가 남는 것과 urgent를 거절한 뒤 high가 남는 것은 이유가 다릅니다. 첫 UPDATE에서는 priority 열을 생략했고, 두 번째 UPDATE는 CHECK에서 실패했습니다.
